# **Random Forest Classification test**

In [ ]:
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURAÇÃO ---
OUTPUT_DIR = 'aumentado' # O diretório com suas imagens 512x512
RANDOM_STATE = 42 # Para reprodutibilidade
# --------------------

def load_data_and_extract_features(data_dir):
    """Carrega as imagens e extrai features (pixels achatados) e labels."""
    data = []
    labels = []
    
    # Percorre cada subdiretório (classe/letra)
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        
        if not os.path.isdir(class_path):
            continue
            
        print(f"Carregando imagens da classe: {class_name}")
        
        for filename in os.listdir(class_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_path, filename)
                
                # Lê a imagem
                img = cv2.imread(img_path)
                
                if img is not None:
                    # Verifica se a imagem é 512x512x3 (esperado)
                    if img.shape == (512, 512, 3):
                        # 1. Extração de Feature (Método Básico: Achatamento de Pixels)
                        # Uma imagem 512x512x3 vira um vetor de 512*512*3 = 786,432 features.
                        feature_vector = img.flatten() 
                        
                        data.append(feature_vector)
                        labels.append(class_name)
    
    return np.array(data), np.array(labels)

# 1. Carregar e Processar
X, y = load_data_and_extract_features(OUTPUT_DIR)
print(f"\nTotal de amostras carregadas: {X.shape[0]}")
print(f"Dimensão de cada feature (pixels): {X.shape[1]}")

# 2. Dividir em Treino e Teste
# Usamos 80% para treino e 20% para teste (para avaliar a generalização)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Amostras de Treino: {X_train.shape[0]}, Amostras de Teste: {X_test.shape[0]}")

In [ ]:
# 3. Treinamento do Random Forest
print("\nIniciando o treinamento do Random Forest...")
# Usamos 100 árvores (n_estimators=100)
rf_model = RandomForestClassifier(
    n_estimators=200, 
    random_state=RANDOM_STATE, 
    n_jobs=-1, # Usa todos os núcleos da CPU
    verbose=1
)

rf_model.fit(X_train, y_train)
print("Treinamento concluído.")

In [ ]:
# 4. Predição no conjunto de Teste
print("Fazendo predições no conjunto de teste...")
y_pred = rf_model.predict(X_test)

# 5. Métricas de Desempenho
class_labels = np.unique(y)

## A. Acurácia Global
accuracy = accuracy_score(y_test, y_pred)
print(f"\n## Acurácia Global (Accuracy): {accuracy:.4f}")

## B. Relatório de Classificação (Precision, Recall, F1-Score)
print("\n## Relatório de Classificação (por Classe):")
print(classification_report(y_test, y_pred, target_names=class_labels, zero_division=0))

## C. Matriz de Confusão
cm = confusion_matrix(y_test, y_pred, labels=class_labels)

# Visualização da Matriz de Confusão
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_labels, 
    yticklabels=class_labels
)
plt.title('Matriz de Confusão - Random Forest (Pixels Achados)')
plt.ylabel('Rótulo Verdadeiro')
plt.xlabel('Rótulo Predito')
plt.show()